In [1]:
import os
os.chdir('..')
print(os.getcwd())

d:\pythonProject\IC Lab\Gait_analysis\pyskl


In [ ]:
from colab.SSO import SSO
import pandas as pd

def before_after(logs):
    before_list = []
    after_list = []
    for i in logs:
        SSO_reader = SSO()
        SSO_reader.load_result(i)
        before_log = SSO_reader.ckpt[0][0]['message']
        before = (
            0,
            0,
            before_log['train_time'],
            before_log['test_time'],  # 確保 test_cost 存在
            before_log['val_acc_4c'],
            before_log['test_acc_4c'],
            before_log['test_acc_3c'],
            before_log['test_acc_2c'],
            before_log['f1'],
            before_log['f1_3c'],
            before_log['f1_4c'],
            before_log['precision'],
            before_log['recall'],
            SSO_reader.ckpt[0][0]['x'][0] # params
        )

        # 取得最佳解的紀錄（最佳世代 genBest，最佳解 gBest）
        after_log = SSO_reader.ckpt[SSO_reader.genBest][SSO_reader.gBest]['message']
        after = (
            SSO_reader.genBest,
            SSO_reader.gBest,
            after_log['train_time'],
            after_log['test_time'],
            after_log['val_acc_4c'],
            after_log['test_acc_4c'],
            after_log['test_acc_3c'],
            after_log['test_acc_2c'],
            after_log['f1'],
            after_log['f1_4c'],
            after_log['f1_3c'],
            after_log['precision'],
            after_log['recall'],
            SSO_reader.ckpt[SSO_reader.genBest][SSO_reader.gBest]['x'][0] # params
        )
        print(f"Fitness : {SSO_reader.ckpt[SSO_reader.genBest][SSO_reader.gBest]['x'][1]}")
        print(f"params : {SSO_reader.ckpt[SSO_reader.genBest][SSO_reader.gBest]['x'][0]}")
        print(before)
        print(after)
        before_list.append(before)
        after_list.append(after)
    return before_list, after_list

In [ ]:
# SSO_reader = SSO()
# SSO_reader.load_result(r"C:\Users\User\Downloads\drunk_4_789_ggen15_gsol2_searchtime60477.525334358215_fec2a3da.pkl")
# print(SSO_reader.ckpt[SSO_reader.genBest][SSO_reader.gBest])

❌ Unable to extract ggen and gsol from filename.


KeyError: 0

# 小樣本試驗

In [3]:
import os
import glob

# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\sso_result"
# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\tmp(droplast)"
# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\sso_消融實驗"
# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\sso_消融2"
# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\sso_消融458"

results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\sso_tri"
# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\SSO_消融458(主要數據)"
# results_dir = r"D:\pythonProject\IC Lab\Gait_analysis\pyskl\SSO_tri2"
data = 'drunk'
split = 8
# hyp (789, 189, 129, 123)  348 458
hyp = 348

# 定義符合條件的檔案名稱模式
pattern = os.path.join(results_dir, f"{data}_{split}_{hyp}_*")

# 搜尋所有符合條件的檔案
logs = glob.glob(pattern)
before_list, after_list = before_after(logs)


🔄 Loading from saved checkpoint...
📂 File ggen: 10, gsol: 5
Fitness : 0.8156028368794326
params : (2.0, 188.0, 3.0, 2.0, 3.0, 0.0, 20.0, 0.0, 24.0, 10.0, 92.0, 71.0, 50.0)
(0, 0, 123.0400800704956, 0.47888684272766113, 0.773936170212766, 0.7486111111111111, 0.8625, 0.9701388888888889, 0.9692197566213314, 1.0, 0.9402777777777778, (4.0, 64.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 100.0, 5.0, 90.0, 60.0, 50.0))
(10, 5, 222.59132933616638, 0.7334799766540527, 0.8156028368794326, 0.7770833333333333, 0.8722222222222222, 0.9819444444444444, 0.9818435754189944, 0.9873595505617978, 0.9763888888888889, (2.0, 188.0, 3.0, 2.0, 3.0, 0.0, 20.0, 0.0, 24.0, 10.0, 92.0, 71.0, 50.0))

🔄 Loading from saved checkpoint...
📂 File ggen: 10, gsol: 7
Fitness : 0.8359929078014184
params : (2.0, 129.0, 1.0, 3.0, 0.0, 0.0, 17.0, 23.0, 45.0, 30.0, 75.0, 49.0, 22.0)
(0, 0, 157.38249349594116, 0.4749317169189453, 0.7854609929078015, 0.7833333333333333, 0.8708333333333333, 0.975, 0.9745403111739745, 0.9927953890489913, 0.95

In [ ]:
columns = ["Stage", "genBest", "gBest", "train_time", "test_time", "val_acc_4c", "test_acc_4c", "test_acc_3c", "test_acc_2c", "f1", "f1_4c", "f1_3c", "precision", "recall", "params"]

df_before = pd.DataFrame(before_list, columns=columns[1:])
df_before.insert(0, "Stage", "Before")

df_after = pd.DataFrame(after_list, columns=columns[1:])
df_after.insert(0, "Stage", "After")

# 合併兩個DataFrame
df_final = pd.concat([df_before, df_after], ignore_index=True)

# 保存為 CSV 檔案
csv_path = "before_after_results.csv"
df_final.to_csv(csv_path, index=False)
print("Done")